# sparse_actions - calibration scatter plots

Asked-vs-actual calibration for the toy complexity ladder (rung1-rung4, Qwen-1.5B) and the
new realistic on-policy refusal run (Llama-3.1-8B). Reads the committed `outputs/**/eval` CSVs;
figures are saved to `notebooks/figures/`.

- **blue = held-out (test)** rates, never seen in training
- **orange = seen in training** - only the discrete-grid rung1 has these; the other rungs and
  the refusal run were trained on a *continuous* range of rates, so every eval point is held-out.

Run from the repo root (`outputs/` and `notebooks/figures/` are resolved relative to it).

In [ ]:
%matplotlib inline
import json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.rcParams.update({'figure.dpi':120,'savefig.dpi':150,'font.size':11,
                     'axes.grid':True,'grid.alpha':0.3,'grid.linestyle':':','axes.axisbelow':True})
OUT = Path('outputs'); FIG = Path('notebooks/figures'); FIG.mkdir(parents=True, exist_ok=True)

# colorblind-safe (Okabe-Ito): 2-class blue/orange + a fixed 5-series order
C_SEEN = '#E69F00'   # orange - rates the model SAW during training
C_TEST = '#0072B2'   # blue   - held-out (test) rates
OKABE  = ['#0072B2', '#E69F00', '#009E73', '#D55E00', '#CC79A7']

def idline(ax, lo, hi):
    ax.plot([lo, hi], [lo, hi], '--', color='0.5', lw=1.1, zorder=1)

def logfmt(ax, lo, hi):
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect('equal')

def load_cal(rel):
    """Load a calibration CSV -> columns include target_p, realized_p, held_out (bool)."""
    df = pd.read_csv(OUT / rel)
    df['held_out'] = df['held_out'].astype(str).str.strip().str.lower().eq('true')
    return df

def panel(ax, df, title, lo=5e-5, hi=1):
    idline(ax, lo, hi)
    for flag, color, lab in [(False, C_SEEN, 'seen in training'), (True, C_TEST, 'held-out (test)')]:
        sub = df[df.held_out == flag]
        if len(sub):
            ax.scatter(sub.target_p, sub.realized_p, s=55, color=color,
                       edgecolor='w', lw=.6, zorder=3, label=lab)
    logfmt(ax, lo, hi); ax.set_title(title, fontsize=10)

print('setup ok - outputs entries:', len(list(OUT.glob('*'))))

## 1. Asked vs. actual rate - per rung and the refusal setting

x = the rate we requested, y = the rate the model actually produced; dashed line is perfect
calibration (y = x). rung1 (discrete grid) shows the memorization failure - trained rates land
on the line, in-between rates miss by ~10x. rung2-4 and refusal hug the line down to ~0.001,
with the only miss at the extreme bottom edge of the trained range (a one-sided boundary effect).

In [ ]:
PANELS = [
    ('controllable_rung1/eval/calibration.csv',                     'rung1 - marker  (discrete grid)'),
    ('controllable_rung2/eval/calibration.csv',                     'rung2 - "FLAG:" sentinel'),
    ('controllable_rung3/eval/calibration.csv',                     'rung3 - pick option 3'),
    ('controllable_rung4/eval/calibration.csv',                     'rung4 - all-lowercase'),
    ('refusal_llama_realistic_onpolicy/eval/calibration_curve.csv', 'refusal - Llama comply (realistic)'),
]
fig, axs = plt.subplots(2, 3, figsize=(15, 9.6)); axs = axs.ravel()
for ax, (rel, title) in zip(axs, PANELS):
    panel(ax, load_cal(rel), title)
for i in (2, 3, 4):            # visual bottom of each column (col2's row-2 is the off panel)
    axs[i].set_xlabel('requested (asked) rate')
for i in (0, 3):
    axs[i].set_ylabel('realized (actual) rate')
axs[5].axis('off')
handles = [Line2D([0], [0], marker='o', ls='', mfc=C_SEEN, mec='w', ms=9, label='seen in training'),
           Line2D([0], [0], marker='o', ls='', mfc=C_TEST, mec='w', ms=9, label='held-out (test)'),
           Line2D([0], [0], ls='--', color='0.5', label='perfect calibration (y = x)')]
fig.legend(handles=handles, loc='lower right', bbox_to_anchor=(0.90, 0.14), fontsize=12, frameon=True)
fig.suptitle('Asked vs. actual rate - calibration across the ladder and the refusal setting',
             fontsize=13, y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.98]); fig.savefig(FIG / 'asked_vs_actual_grid.png'); plt.show()

## 2. How accurate is the knob across the rate range

Same data as a ratio: **actual / asked** (1.0 = perfect) vs the requested rate. The knobs sit on
1.0 across most of the range and only blow up at the extreme low edge of what they were trained
on (the boundary artifact). The refusal knob only reaches ~3e-4.

In [ ]:
KNOBS = [
    ('controllable_rung1_cont/eval/calibration.csv',                'rung1 (continuous)'),
    ('controllable_rung2/eval/calibration.csv',                     'rung2 FLAG'),
    ('controllable_rung3/eval/calibration.csv',                     'rung3 option 3'),
    ('controllable_rung4/eval/calibration.csv',                     'rung4 lowercase'),
    ('refusal_llama_realistic_onpolicy/eval/calibration_curve.csv', 'refusal (realistic)'),
]
fig, ax = plt.subplots(figsize=(8.6, 5.6))
for (rel, lab), color in zip(KNOBS, OKABE):
    df = load_cal(rel).sort_values('target_p')
    ax.plot(df.target_p, df.realized_p / df.target_p, '-o', color=color, ms=5, lw=1.6, label=lab)
ax.axhline(1.0, color='0.4', ls='--', lw=1.1)
ax.text(6e-5, 1.06, 'perfect (actual = asked)', color='0.4', fontsize=9)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('requested (asked) rate'); ax.set_ylabel('actual / asked   (1.0 = perfect)')
ax.set_title('How accurate is the knob across the rate range (tail drift at the bottom edge)')
ax.legend(fontsize=9, ncol=2)
fig.tight_layout(); fig.savefig(FIG / 'knob_accuracy_by_rate.png'); plt.show()

## 3. Discrete grid *memorizes*; continuous *generalizes*

The core reason the knob works. Left: trained on 4 fixed rates -> perfect on those (orange),
~10x off on everything else. Right: trained on a continuous range -> a smooth knob that hits
rates it never saw (all points held-out).

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11.5, 5.7), sharex=True, sharey=True)
panel(axs[0], load_cal('controllable_rung1/eval/calibration.csv'),
      'Discrete grid -> memorizes trained rates')
panel(axs[1], load_cal('controllable_rung1_cont/eval/calibration.csv'),
      'Continuous training -> a real knob')
for ax in axs:
    ax.set_xlabel('requested (asked) rate')
axs[0].set_ylabel('realized (actual) rate')
axs[0].legend(loc='upper left', fontsize=9); axs[1].legend(loc='upper left', fontsize=9)
fig.tight_layout(); fig.savefig(FIG / 'rung1_discrete_vs_continuous.png'); plt.show()

## 4. Are the rare compliances real? (rollout quality)

When we force the comply gate on held-out prompts, are the responses genuine on-topic answers?
On AdvBench (harmful) the compliances were hollow - only 9% on-topic. On the realistic (~50%
comply) set they are real answers 80% of the time. (n = 15 held-out prompts, so wide CIs, but
the gap is large.)

In [ ]:
adv = json.loads((OUT / 'refusal_llama_onpolicy/eval/rollout_quality.json').read_text())
rea = json.loads((OUT / 'refusal_llama_realistic_onpolicy/eval/rollout_quality.json').read_text())
metrics = ['b_branch_relevance_rate', 'b_branch_engage_rate']
labels  = ['comply is on-topic\n(relevant to the prompt)', 'comply engages\n(not a refusal)']
adv_v = [adv[m] for m in metrics]; rea_v = [rea[m] for m in metrics]
x = np.arange(len(metrics)); w = 0.38
fig, ax = plt.subplots(figsize=(7.6, 5.2))
b1 = ax.bar(x - w/2, adv_v, w, color=C_SEEN, label='AdvBench (harmful)')
b2 = ax.bar(x + w/2, rea_v, w, color=C_TEST, label='realistic (~50% comply)')
for bars in (b1, b2):
    for r in bars:
        ax.text(r.get_x() + r.get_width()/2, r.get_height() + 0.02, f'{r.get_height():.0%}',
                ha='center', va='bottom', fontsize=10)
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylim(0, 1.10)
ax.set_ylabel('fraction of forced-comply rollouts')
ax.set_title('Rare-compliance rollout quality (forced comply, held-out prompts)', fontsize=11)
ax.legend(fontsize=10, loc='upper left'); ax.grid(axis='x')
fig.tight_layout(); fig.savefig(FIG / 'rollout_quality.png'); plt.show()